# PV+BESS Dispatch Analysis

Deep-dive hourly dispatch diagnostics for the 25-case sizing matrix.

## Structure
| Cell | Content |
|------|----------|
| 1 | Imports + constants |
| 2 | Setup — simulate all 25 cases, cache hourly time-series |
| **Part 1 — Deep dive (3 cases)** | |
| 3 | Chart 1 — Representative week dispatch stack |
| 4 | Chart 2 — Full year SOC heatmap |
| 5 | Chart 3 — Duration curves |
| 6 | Chart 4 — Monthly energy waterfall (Case B) |
| 7 | Chart 5 — Dispatch hourly heatmap (Case B) |
| 8 | Chart 6 — Utilization analysis |
| **Part 2 — Interactive** | |
| 9 | Widget — all 25 cases |

> Site: Scurry County TX | 300 MWac PV | self_consumption dispatch | 200 MW flat load

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))
sys.path.insert(0, str(pathlib.Path('.').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from pvsamlab import System, TrackingMode, Battery, BessDispatch, PvBessSystem

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=0.9)

# ── Site ────────────────────────────────────────────────────────────────────────────────
SITE_LAT       = 33.0278
SITE_LON       = -100.0814
MET_YEAR       = '2017'
PV_TARGET_KWAC = 300_000
PV_DCAC        = 1.35
LOAD_KW        = 200_000          # 200 MW flat dispatch load

# ── Sweep matrix ──────────────────────────────────────────────────────────────────────────
POWER_MW_LIST    = [50, 75, 100, 125, 150]
DURATION_HR_LIST = [1,  2,   4,   6,   8]

# ── Representative cases ───────────────────────────────────────────────────────────────────────
CASE_A = (50,  2)    # best IRR
CASE_B = (100, 4)    # balanced
CASE_C = (150, 8)    # max storage

CASE_LABELS = {
    CASE_A: f'{CASE_A[0]} MW / {CASE_A[1]} hr  (best IRR)',
    CASE_B: f'{CASE_B[0]} MW / {CASE_B[1]} hr  (balanced)',
    CASE_C: f'{CASE_C[0]} MW / {CASE_C[1]} hr  (max storage)',
}

# ── Battery hardware defaults (same as sizing study) ─────────────────────────────────
BATT_KWARGS = dict(
    chemistry='LFP', capex_per_kwh=250.0, capex_per_kw=150.0,
    opex_per_kwh_year=8.0, soc_min=10.0, soc_max=95.0,
    calendar_degradation=2.0,
)

# ── Paths ───────────────────────────────────────────────────────────────────────────────
_HERE       = pathlib.Path('.')
RESULTS_CSV = _HERE / 'pv_bess_sizing_study_results.csv'

# ── Datetime index (8760 h starting 2017-01-01) ────────────────────────────────────────────
DT_IDX = pd.date_range('2017-01-01', periods=8760, freq='h')

# ── Week slices ───────────────────────────────────────────────────────────────────────────
# July 1 = day 182 (1-indexed) → hour 181×24 = 4344
SUMMER_SLICE = slice(4344, 4344 + 7 * 24)   # Jul 1–7
WINTER_SLICE = slice(0, 7 * 24)             # Jan 1–7

print('Imports OK')

In [ ]:
try:
    ts_data
    print(f'Using cached time-series data for {len(ts_data)} cases.')
except NameError:
    ts_data = {}

    _disp = BessDispatch(strategy='self_consumption', can_gridcharge=[0] * 6)
    _cases = [(p, d) for p in POWER_MW_LIST for d in DURATION_HR_LIST]

    for power_mw, duration_hr in tqdm(_cases, desc='Simulating PV+BESS cases'):
        energy_kwh = power_mw * 1000 * duration_hr
        batt = Battery(
            energy_kwh=float(energy_kwh),
            power_kw=float(power_mw * 1000),
            **BATT_KWARGS,
        )
        plant = PvBessSystem(
            lat=SITE_LAT, lon=SITE_LON,
            target_kwac=PV_TARGET_KWAC, target_dcac=PV_DCAC,
            met_year=MET_YEAR, tracking_mode=TrackingMode.SAT,
            battery=batt, dispatch=_disp,
            load_profile=[LOAD_KW] * 8760,
        )
        plant.run()
        out = plant.model.Outputs

        gen_kw   = np.array(list(out.gen))
        batt_kw  = np.array(list(out.batt_power))
        soc_pct  = np.array(list(out.batt_SOC))
        grid_kw  = np.array(list(out.grid_power))

        ts_data[(power_mw, duration_hr)] = {
            'hourly_gen_mw':    gen_kw  / 1000.0,   # PV+batt combined, before load (MW)
            'hourly_batt_mw':   batt_kw / 1000.0,   # + discharge  / – charge (MW)
            'hourly_soc_pct':   soc_pct,             # %
            'hourly_export_mw': grid_kw / 1000.0,   # net grid exchange: gen – load (MW)
            'energy_kwh':       energy_kwh,
        }

    print(f'Done. {len(ts_data)} cases stored in ts_data.')

In [ ]:
# ── Chart 1 — Representative week dispatch stack ────────────────────────────────────────────
# 3 rows × 2 columns (summer / winter)

def _plot_dispatch_panel(ax, ts, sl, panel_title, show_legend=False):
    """Stacked area dispatch plot for a 7-day window."""
    gen_w  = ts['hourly_gen_mw'][sl]
    batt_w = ts['hourly_batt_mw'][sl]
    soc_w  = ts['hourly_soc_pct'][sl]
    exp_w  = ts['hourly_export_mw'][sl]
    dt_w   = DT_IDX[sl]
    hours  = np.arange(len(dt_w))

    # Decompose gen into PV contribution and battery portions
    pv_ac  = gen_w - np.maximum(batt_w, 0)   # PV AC (gen minus what battery added)
    b_dis  = np.maximum(batt_w, 0)            # battery discharge (above zero)
    b_chg  = np.minimum(batt_w, 0)            # battery charge (below zero)

    ax.fill_between(hours, 0, pv_ac,              color='#FFD700', alpha=0.85, label='PV gen')
    ax.fill_between(hours, pv_ac, pv_ac + b_dis,  color='#27AE60', alpha=0.75, label='BESS discharge')
    ax.fill_between(hours, b_chg, 0,              color='#E74C3C', alpha=0.70, label='BESS charge')
    ax.plot(hours, exp_w, color='#2C3E50', linewidth=1.5, label='Grid export', zorder=5)
    ax.axhline(300, color='black', linestyle='--', linewidth=0.8, alpha=0.35, label='300 MW nameplate')

    # Secondary axis: SOC
    ax2 = ax.twinx()
    ax2.plot(hours, soc_w, color='grey', linestyle='--', linewidth=1.0, alpha=0.6)
    ax2.set_ylim(0, 115)
    ax2.set_ylabel('SOC (%)', color='grey', fontsize=7)
    ax2.tick_params(axis='y', labelcolor='grey', labelsize=7)

    day_ticks = np.arange(0, len(hours), 24)
    ax.set_xticks(day_ticks)
    ax.set_xticklabels([dt_w[i].strftime('%b %d') for i in day_ticks], fontsize=7)
    ax.set_title(panel_title, fontsize=8)
    ax.set_ylabel('Power (MW)', fontsize=8)
    if show_legend:
        ax.legend(loc='upper right', fontsize=6, ncol=3)


fig, axes = plt.subplots(3, 2, figsize=(20, 14))
fig.suptitle(
    'Representative Week Dispatch — 300 MWac PV + BESS\n'
    'self_consumption dispatch | 200 MW flat load',
    fontsize=12,
)

for row, case in enumerate([CASE_A, CASE_B, CASE_C]):
    ts = ts_data[case]
    lbl = CASE_LABELS[case]
    _plot_dispatch_panel(
        axes[row, 0], ts, SUMMER_SLICE,
        f'Summer week (Jul 1–7)  |  {lbl}',
        show_legend=(row == 0),
    )
    _plot_dispatch_panel(
        axes[row, 1], ts, WINTER_SLICE,
        f'Winter week (Jan 1–7)  |  {lbl}',
    )

plt.tight_layout()
_PNG = _HERE / 'dispatch_weekly.png'
plt.savefig(_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {_PNG}')

In [ ]:
# ── Chart 2 — Full year SOC heatmap (3 cases, shared colorbar) ────────────────────────

MONTH_START_DAYS  = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
MONTH_ABBR        = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Full Year State of Charge Heatmap — Scurry County TX 2017', fontsize=11)
_im = None

for col, case in enumerate([CASE_A, CASE_B, CASE_C]):
    ts     = ts_data[case]
    energy = ts['energy_kwh']
    soc_2d = ts['hourly_soc_pct'][:8760].reshape(365, 24).T   # (24 hrs × 365 days)

    total_dis_kwh  = np.maximum(ts['hourly_batt_mw'], 0).sum() * 1000   # MW·h → kWh
    avg_daily_cyc  = total_dis_kwh / (energy * 365)

    ax = axes[col]
    _im = ax.pcolormesh(
        np.arange(365), np.arange(24), soc_2d,
        cmap='RdYlGn', vmin=0, vmax=100,
    )
    for d, m in zip(MONTH_START_DAYS, MONTH_ABBR):
        if d > 0:
            ax.axvline(d, color='grey', linewidth=0.5, alpha=0.5)
        ax.text(d + 1, 23.5, m, fontsize=6, color='black', va='top', ha='left')

    ax.set_xlabel('Day of Year')
    ax.set_ylabel('Hour of Day' if col == 0 else '')
    ax.set_title(f'{CASE_LABELS[case]}\navg daily cycles = {avg_daily_cyc:.2f}', fontsize=9)
    ax.set_yticks([0, 6, 12, 18, 23])
    ax.set_yticklabels(['00:00', '06:00', '12:00', '18:00', '23:00'], fontsize=7)

fig.colorbar(_im, ax=axes.tolist(), shrink=0.55, label='SOC (%)', pad=0.02)
plt.tight_layout()
_PNG = _HERE / 'soc_heatmap.png'
plt.savefig(_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {_PNG}')

In [ ]:
# ── Chart 3 — Duration curves ─────────────────────────────────────────────────────────────────────

# PV-only: derive from any case (all share the same PV plant / weather)
_ref  = ts_data[CASE_A]
pv_only_mw = _ref['hourly_gen_mw'] - _ref['hourly_batt_mw']   # PV AC without battery

def _dur(arr):
    return np.sort(arr)[::-1]

x_hrs = np.arange(8760)
pv_dc  = _dur(pv_only_mw)
dc_a   = _dur(ts_data[CASE_A]['hourly_gen_mw'])
dc_b   = _dur(ts_data[CASE_B]['hourly_gen_mw'])
dc_c   = _dur(ts_data[CASE_C]['hourly_gen_mw'])

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(x_hrs, pv_dc, color='grey',    linewidth=2.0, label='PV only')
ax.plot(x_hrs, dc_a,  color='#AED6F1', linewidth=1.5, label=f'Case A  {CASE_LABELS[CASE_A]}')
ax.plot(x_hrs, dc_b,  color='#2980B9', linewidth=1.5, label=f'Case B  {CASE_LABELS[CASE_B]}')
ax.plot(x_hrs, dc_c,  color='#1A5276', linewidth=2.0, label=f'Case C  {CASE_LABELS[CASE_C]}')

ax.fill_between(x_hrs, pv_dc, dc_c, alpha=0.10, color='#3498DB',
                label='BESS net contribution (PV-only vs Case C)')

ax.axvline(4380, color='black', linestyle='--', linewidth=1.0, alpha=0.6)
ax.axhline(300,  color='black', linestyle='--', linewidth=0.8, alpha=0.3)

# CF annotations at 4380h
y_top = max(pv_dc[0], dc_c[0]) * 1.02
ax.text(4390, y_top * 0.97, '← 4380 h (CF = 50 %)', fontsize=8)
ax.text(150, 305, '300 MW nameplate', fontsize=8, alpha=0.6)

for dc, lbl, col in [
    (pv_dc, 'PV',     'grey'),
    (dc_a,  'Case A', '#2980B9'),
    (dc_b,  'Case B', '#2980B9'),
    (dc_c,  'Case C', '#1A5276'),
]:
    cf = dc[4380] / 300 * 100
    ax.annotate(
        f'{lbl}: {cf:.0f} % CF',
        xy=(4380, dc[4380]), xytext=(4500, dc[4380] - 5),
        fontsize=7, color=col,
        arrowprops=dict(arrowstyle='-', color=col, lw=0.8),
    )

ax.set_xlabel('Hours (ranked, descending)')
ax.set_ylabel('Power (MW)')
ax.set_title('Duration Curves — PV-only vs PV+BESS Cases')
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(0, 8760)

plt.tight_layout()
_PNG = _HERE / 'duration_curves.png'
plt.savefig(_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {_PNG}')

In [ ]:
# ── Chart 4 — Monthly energy waterfall (Case B) ────────────────────────────────────────────

ts = ts_data[CASE_B]
df_ts = pd.DataFrame({
    'gen_mw':    ts['hourly_gen_mw'],
    'batt_mw':   ts['hourly_batt_mw'],
    'soc_pct':   ts['hourly_soc_pct'],
    'export_mw': ts['hourly_export_mw'],
}, index=DT_IDX)

# Monthly totals (GWh)
pv_mon  = (df_ts['gen_mw'] - df_ts['batt_mw'].clip(lower=0)).resample('ME').sum() / 1000
dis_mon = df_ts['batt_mw'].clip(lower=0).resample('ME').sum() / 1000
chg_mon = (-df_ts['batt_mw'].clip(upper=0)).resample('ME').sum() / 1000
exp_mon = df_ts['export_mw'].resample('ME').sum() / 1000
soc_mon = df_ts['soc_pct'].resample('ME').mean()

months = [m.strftime('%b') for m in pv_mon.index]
x = np.arange(len(months))
w = 0.20

fig, ax1 = plt.subplots(figsize=(14, 6))
fig.suptitle(f'Monthly Energy Balance — Case B: {CASE_LABELS[CASE_B]}', fontsize=11)

b1 = ax1.bar(x - 1.5*w, pv_mon,   w, color='#FFD700', alpha=0.85, label='PV gen (GWh)')
b2 = ax1.bar(x - 0.5*w, -chg_mon, w, color='#E74C3C', alpha=0.80, label='BESS charge (GWh)')
b3 = ax1.bar(x + 0.5*w, dis_mon,  w, color='#27AE60', alpha=0.80, label='BESS discharge (GWh)')
b4 = ax1.bar(x + 1.5*w, exp_mon,  w, color='#2C3E50', alpha=0.85, label='Net export (GWh)')

for bar_grp, is_neg in [(b1, False), (b3, False), (b4, False), (b2, True)]:
    for bar in bar_grp:
        h = bar.get_height()
        if abs(h) > 1.0:
            ypos = h + 0.5 if not is_neg else h - 0.5
            va   = 'bottom' if not is_neg else 'top'
            ax1.text(bar.get_x() + bar.get_width() / 2, ypos,
                     f'{abs(h):.0f}', ha='center', va=va, fontsize=5.5)

ax1.set_xticks(x)
ax1.set_xticklabels(months)
ax1.set_ylabel('Energy (GWh)')
ax1.axhline(0, color='black', linewidth=0.5)

ax2 = ax1.twinx()
ax2.plot(x, soc_mon, color='grey', marker='o', markersize=4,
         linewidth=1.5, linestyle='--', alpha=0.75, label='Avg SOC %')
ax2.set_ylabel('Avg SOC (%)', color='grey', fontsize=9)
ax2.set_ylim(0, 100)
ax2.tick_params(axis='y', labelcolor='grey')

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=8, loc='upper left')

plt.tight_layout()
_PNG = _HERE / 'monthly_waterfall.png'
plt.savefig(_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {_PNG}')

In [ ]:
# ── Chart 5 — Dispatch heatmap hour × day (Case B) ───────────────────────────────────

ts = ts_data[CASE_B]
# Reshape net export to (24 hr × 365 day); hour 0 = midnight at top (invert_yaxis)
export_2d = ts['hourly_export_mw'][:8760].reshape(365, 24).T   # (24, 365)
vmax = float(np.percentile(np.abs(ts['hourly_export_mw']), 99))

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.pcolormesh(
    np.arange(365), np.arange(24), export_2d,
    cmap='RdBu_r', vmin=-vmax, vmax=vmax,
)
fig.colorbar(im, ax=ax, label='Net export (MW)', shrink=0.8, pad=0.01)

for d, m in zip(MONTH_START_DAYS, MONTH_ABBR):
    if d > 0:
        ax.axvline(d, color='white', linewidth=0.9, alpha=0.7)
    ax.text(d + 1, 22.5, m, fontsize=6.5, color='white', va='top', ha='left')

ax.invert_yaxis()   # hour 0 = midnight at top
ax.set_xlabel('Day of Year')
ax.set_ylabel('Hour of Day')
ax.set_title(
    f'Hourly Dispatch Heatmap — Case B: {CASE_LABELS[CASE_B]}\n'
    'Red = exporting to grid  |  Blue = net import (battery charging from excess PV)',
    fontsize=9,
)
ax.set_yticks([0, 6, 12, 18, 23])
ax.set_yticklabels(['00:00', '06:00', '12:00', '18:00', '23:00'])

plt.tight_layout()
_PNG = _HERE / 'dispatch_hourly_heatmap.png'
plt.savefig(_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {_PNG}')

In [ ]:
# ── Chart 6 — Utilization analysis (3 cases) ─────────────────────────────────────────────

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Battery Utilization Analysis — Three Representative Cases', fontsize=11)

_palette = {CASE_A: '#AED6F1', CASE_B: '#2980B9', CASE_C: '#1A5276'}

for case in [CASE_A, CASE_B, CASE_C]:
    ts    = ts_data[case]
    color = _palette[case]
    label = CASE_LABELS[case]
    energy_kwh = ts['energy_kwh']
    batt_mw    = ts['hourly_batt_mw']

    # Daily cycle depth
    daily_dis_kwh = np.maximum(batt_mw * 1000, 0).reshape(365, 24).sum(axis=1)
    cycle_depth   = daily_dis_kwh / energy_kwh

    ax_l.hist(cycle_depth, bins=40, alpha=0.5, color=color, label=label)
    ax_l.axvline(cycle_depth.mean(), color=color, linewidth=1.5, linestyle='--',
                 label=f'Mean = {cycle_depth.mean():.2f}')

    # Hourly mean SOC ± 1 std
    soc_2d   = ts['hourly_soc_pct'][:8760].reshape(365, 24)
    mean_soc = soc_2d.mean(axis=0)
    std_soc  = soc_2d.std(axis=0)
    hrs = np.arange(24)
    ax_r.plot(hrs, mean_soc, color=color, linewidth=1.5, label=label)
    ax_r.fill_between(hrs, mean_soc - std_soc, mean_soc + std_soc,
                      color=color, alpha=0.15)

ax_l.set_xlabel('Daily Cycle Depth (fraction of energy capacity)')
ax_l.set_ylabel('Number of Days')
ax_l.set_title('Daily Cycle Depth Distribution')
ax_l.legend(fontsize=7)

ax_r.set_xlabel('Hour of Day')
ax_r.set_ylabel('Mean SOC (%)')
ax_r.set_title('Average SOC by Hour of Day (±1 std dev shaded)')
ax_r.set_xticks(range(0, 24, 3))
ax_r.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 3)], fontsize=8)
ax_r.legend(fontsize=7)

plt.tight_layout()
_PNG = _HERE / 'utilization_analysis.png'
plt.savefig(_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {_PNG}')

In [ ]:
# ── Part 2 — Interactive widget (all 25 cases) ────────────────────────────────────────────

_fin_df = pd.read_csv(RESULTS_CSV)

power_dd    = widgets.Dropdown(
    options=[50, 75, 100, 125, 150], value=100,
    description='Power (MW):', style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px'),
)
duration_dd = widgets.Dropdown(
    options=[1, 2, 4, 6, 8], value=4,
    description='Duration (hr):', style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px'),
)
out_widget = widgets.Output()


def _redraw(change=None):
    pw = power_dd.value
    dr = duration_dd.value
    ts = ts_data[(pw, dr)]
    row = _fin_df.query('power_mw == @pw and duration_hr == @dr').iloc[0]
    energy_kwh = ts['energy_kwh']

    with out_widget:
        clear_output(wait=True)

        fig, axes = plt.subplots(2, 2, figsize=(18, 10))
        fig.suptitle(
            f'Dispatch Analysis  —  {pw} MW / {dr} hr  ({pw*dr:,} MWh)',
            fontsize=12, fontweight='bold',
        )

        # ── Panel 1: Summer week dispatch ─────────────────────────────────────────────────────
        sl     = SUMMER_SLICE
        gen_w  = ts['hourly_gen_mw'][sl]
        batt_w = ts['hourly_batt_mw'][sl]
        soc_w  = ts['hourly_soc_pct'][sl]
        exp_w  = ts['hourly_export_mw'][sl]
        dt_w   = DT_IDX[sl]
        hours  = np.arange(len(dt_w))
        pv_ac  = gen_w - np.maximum(batt_w, 0)
        b_dis  = np.maximum(batt_w, 0)
        b_chg  = np.minimum(batt_w, 0)

        ax = axes[0, 0]
        ax.fill_between(hours, 0, pv_ac,             color='#FFD700', alpha=0.85, label='PV gen')
        ax.fill_between(hours, pv_ac, pv_ac + b_dis, color='#27AE60', alpha=0.75, label='BESS discharge')
        ax.fill_between(hours, b_chg, 0,             color='#E74C3C', alpha=0.70, label='BESS charge')
        ax.plot(hours, exp_w, color='#2C3E50', linewidth=1.5, label='Grid export', zorder=5)
        ax.axhline(300, color='black', linestyle='--', linewidth=0.7, alpha=0.35)
        ax2 = ax.twinx()
        ax2.plot(hours, soc_w, color='grey', linestyle='--', linewidth=1.0, alpha=0.6)
        ax2.set_ylim(0, 115)
        ax2.set_ylabel('SOC (%)', color='grey', fontsize=7)
        ax2.tick_params(axis='y', labelcolor='grey', labelsize=7)
        day_ticks = np.arange(0, len(hours), 24)
        ax.set_xticks(day_ticks)
        ax.set_xticklabels([dt_w[i].strftime('%b %d') for i in day_ticks], fontsize=7)
        ax.set_title('Summer Week Dispatch (Jul 1–7)', fontsize=9)
        ax.set_ylabel('Power (MW)', fontsize=8)
        ax.legend(fontsize=6, loc='upper right', ncol=2)

        # ── Panel 2: Full year SOC heatmap ─────────────────────────────────────────────────────
        ax = axes[0, 1]
        soc_2d = ts['hourly_soc_pct'][:8760].reshape(365, 24).T
        im = ax.pcolormesh(np.arange(365), np.arange(24), soc_2d,
                           cmap='RdYlGn', vmin=0, vmax=100)
        fig.colorbar(im, ax=ax, label='SOC (%)', shrink=0.85, pad=0.01)
        for d, m in zip(MONTH_START_DAYS, MONTH_ABBR):
            if d > 0:
                ax.axvline(d, color='white', linewidth=0.5, alpha=0.5)
            ax.text(d + 1, 23.5, m[0], fontsize=6, color='white', va='top')
        ax.invert_yaxis()
        ax.set_xlabel('Day of Year')
        ax.set_ylabel('Hour of Day')
        ax.set_title('Full Year SOC Heatmap', fontsize=9)
        ax.set_yticks([0, 6, 12, 18, 23])
        ax.set_yticklabels(['00:00', '06:00', '12:00', '18:00', '23:00'], fontsize=7)

        # ── Panel 3: Duration curve ───────────────────────────────────────────────────────────
        ax = axes[1, 0]
        pv_only = ts['hourly_gen_mw'] - ts['hourly_batt_mw']
        pv_dc   = np.sort(pv_only)[::-1]
        sel_dc  = np.sort(ts['hourly_gen_mw'])[::-1]
        x_h     = np.arange(8760)
        ax.plot(x_h, pv_dc,  color='grey',    linewidth=1.5, label='PV only')
        ax.plot(x_h, sel_dc, color='#2980B9', linewidth=1.5, label=f'{pw} MW / {dr} hr')
        ax.fill_between(x_h, pv_dc, sel_dc,
                        where=(sel_dc > pv_dc), alpha=0.20, color='#27AE60', label='BESS adds')
        ax.fill_between(x_h, pv_dc, sel_dc,
                        where=(sel_dc < pv_dc), alpha=0.15, color='#E74C3C', label='BESS reduces')
        ax.axvline(4380, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.axhline(300,  color='black', linestyle='--', linewidth=0.8, alpha=0.3)
        ax.set_xlabel('Hours (ranked)')
        ax.set_ylabel('Power (MW)')
        ax.set_title('Duration Curve vs PV-only', fontsize=9)
        ax.legend(fontsize=7)
        ax.set_xlim(0, 8760)

        # ── Panel 4: Daily cycle depth histogram ─────────────────────────────────────────────
        ax = axes[1, 1]
        batt_mw     = ts['hourly_batt_mw']
        daily_dis   = np.maximum(batt_mw * 1000, 0).reshape(365, 24).sum(axis=1)
        cycle_depth = daily_dis / energy_kwh
        ax.hist(cycle_depth, bins=40, color='#2980B9', alpha=0.75)
        ax.axvline(cycle_depth.mean(), color='red', linewidth=1.5, linestyle='--',
                   label=f'Mean: {cycle_depth.mean():.2f}')
        ax.set_xlabel('Daily Cycle Depth (fraction of capacity)')
        ax.set_ylabel('Days')
        ax.set_title('Daily Cycle Depth Distribution', fontsize=9)
        ax.legend(fontsize=8)

        plt.tight_layout()
        plt.show()

        # Summary stats box
        annual_dis_mwh = float(np.maximum(batt_mw, 0).sum() * 1000)
        print(
            f"\n  Selected case:      {pw:>3} MW / {dr} hr  ({pw*dr:,} MWh)\n"
            f"  Annual discharge:  {annual_dis_mwh:>9,.0f} MWh/yr\n"
            f"  Avg daily cycles:  {cycle_depth.mean():>9.2f}\n"
            f"  LCOS:              {row['lcos_usd_per_kwh']:>9.3f} $/kWh\n"
            f"  IRR:               {row['irr_pct']:>9.2f} %\n"
            f"  NPV:               {row['npv_usd']/1e6:>9.1f} $M"
        )


power_dd.observe(_redraw, names='value')
duration_dd.observe(_redraw, names='value')

display(widgets.HBox([power_dd, duration_dd]))
display(out_widget)
_redraw()